In [ ]:
# Set up Step 3 for the current full pipeline.
#
# Upload config.py or config_positional_context_full_drive.py before running.
# Input files are read from cfg.STEP3_INPUT_DIR.
# Outputs are written to cfg.STEP3_CANONICAL_OUTPUT_DIR or
# cfg.STEP3_DIVERSE_OUTPUT_DIR.

import os
import sys
import shutil
import importlib
import json
import random
import hashlib
import re

from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

from google.colab import files, drive

drive.mount("/content/drive")


PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

print("Upload config.py / config_positional_context_full_drive.py")
uploaded_config = files.upload()

config_upload_name = None
for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

shutil.copy(
    os.path.join("/content", config_upload_name),
    os.path.join(PILOT_ROOT, "config.py"),
)


if "pilot_full.config" in sys.modules:
    del sys.modules["pilot_full.config"]

import pilot_full.config as cfg
importlib.reload(cfg)

os.makedirs(cfg.STEP3_INPUT_DIR, exist_ok=True)
os.makedirs(cfg.STEP3_CANONICAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(cfg.STEP3_DIVERSE_OUTPUT_DIR, exist_ok=True)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("STEP3_INPUT_DIR:", cfg.STEP3_INPUT_DIR)
print("STEP3_CANONICAL_OUTPUT_DIR:", cfg.STEP3_CANONICAL_OUTPUT_DIR)
print("STEP3_DIVERSE_OUTPUT_DIR:", cfg.STEP3_DIVERSE_OUTPUT_DIR)
print("RUN_MODE:", cfg.RUN_MODE)

print("\nStep 2 input files:")
for name in sorted(os.listdir(cfg.STEP3_INPUT_DIR)):
    if name.endswith("_chunk_relations.json"):
        print("-", name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Upload config.py / config_positional_context_full_drive.py


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py

Loaded config:
PILOT_ROOT: /content/pilot_full
STEP3_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step2_rel
STEP3_CANONICAL_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_can
STEP3_DIVERSE_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div
RUN_MODE: diverse

Step 2 input files:
- FloorPlan10_chunk_relations.json
- FloorPlan11_chunk_relations.json
- FloorPlan12_chunk_relations.json
- FloorPlan13_chunk_relations.json
- FloorPlan14_chunk_relations.json
- FloorPlan15_chunk_relations.json
- FloorPlan16_chunk_relations.json
- FloorPlan17_chunk_relations.json
- FloorPlan18_chunk_relations.json
- FloorPlan19_chunk_relations.json
- FloorPlan1_chunk_relations.json
- FloorPlan201_chunk_relations.json
- FloorPlan202_chunk_relations.json
- FloorPlan203_ch

In [ ]:
import pilot_full.config as cfg
importlib.reload(cfg)

from pilot_full.config import (
    RANDOM_SEED,
    SCENES,

    STEP3_INPUT_DIR,
    STEP3_CANONICAL_OUTPUT_DIR,
    STEP3_DIVERSE_OUTPUT_DIR,

    EXPERIMENT_NAME,

    MAX_PARAGRAPHS_PER_CLUSTER,

    MIN_RELATIONS_PER_PARAGRAPH,
    TARGET_RELATIONS_PER_PARAGRAPH,
    MAX_RELATIONS_PER_PARAGRAPH,

    USE_PREFERRED_RELATIONS_FIRST,

    ALLOW_NEAR_IN_TEXT,
    MAX_NEAR_SENTENCES_PER_PARAGRAPH,

    ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT,

    INCLUDE_NONE_RELATION_PAIRS,
    NONE_RELATION_LABEL,

    TARGET_OBJECT_COVERAGE_RATIO,
    MAX_OBJECTS_PER_PARAGRAPH,

    NEW_OBJECT_BONUS,
    TWO_NEW_OBJECT_BONUS,

    ALLOW_REDUNDANT_RELATIONS,
    MAX_REDUNDANT_RELATIONS,

    RELATION_SELECTION_JITTER,
    REUSED_RELATION_PENALTY,

    CANONICAL_MODE,
    DIVERSE_MODE,
    RUN_MODE,

    INTRO_TEMPLATES,
    RELATION_TEMPLATES,
    CANONICAL_TEMPLATE,
    TEXT_RELATION_PRIORITY,

    INVERSE_TO_NATURAL,
    NATURAL_INVERSE_SWAP_RELATIONS,
    INVERSE_LABEL_MAP,
)

random.seed(RANDOM_SEED)

os.makedirs(STEP3_CANONICAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(STEP3_DIVERSE_OUTPUT_DIR, exist_ok=True)

print("STEP3_INPUT_DIR:", STEP3_INPUT_DIR)
print("STEP3_CANONICAL_OUTPUT_DIR:", STEP3_CANONICAL_OUTPUT_DIR)
print("STEP3_DIVERSE_OUTPUT_DIR:", STEP3_DIVERSE_OUTPUT_DIR)
print("RUN_MODE:", RUN_MODE)

STEP3_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step2_rel
STEP3_CANONICAL_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_can
STEP3_DIVERSE_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div
RUN_MODE: diverse


In [ ]:
print("Step 3 parameters are controlled by /content/pilot_full/config.py.")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("RUN_MODE:", RUN_MODE)
print("MAX_PARAGRAPHS_PER_CLUSTER:", MAX_PARAGRAPHS_PER_CLUSTER)
print("MIN_RELATIONS_PER_PARAGRAPH:", MIN_RELATIONS_PER_PARAGRAPH)
print("TARGET_RELATIONS_PER_PARAGRAPH:", TARGET_RELATIONS_PER_PARAGRAPH)
print("MAX_RELATIONS_PER_PARAGRAPH:", MAX_RELATIONS_PER_PARAGRAPH)
print("TARGET_OBJECT_COVERAGE_RATIO:", TARGET_OBJECT_COVERAGE_RATIO)
print("MAX_OBJECTS_PER_PARAGRAPH:", MAX_OBJECTS_PER_PARAGRAPH)

Step 3 parameters are controlled by /content/pilot_full/config.py.
EXPERIMENT_NAME: step3_relation_classification_preserve_text_direction
RUN_MODE: diverse
MAX_PARAGRAPHS_PER_CLUSTER: 4
MIN_RELATIONS_PER_PARAGRAPH: 6
TARGET_RELATIONS_PER_PARAGRAPH: 16
MAX_RELATIONS_PER_PARAGRAPH: 18
TARGET_OBJECT_COVERAGE_RATIO: 0.75
MAX_OBJECTS_PER_PARAGRAPH: 12


In [ ]:
# Relation templates are loaded from pilot_full.config

print("Loaded relation templates from config:")

for relation, templates in RELATION_TEMPLATES.items():
    print(f"- {relation}: {len(templates)} templates; canonical = {templates[0]}")

print("\nLoaded relation priority from config:")
print(TEXT_RELATION_PRIORITY)

Loaded relation templates from config:
- in: 10 templates; canonical = {subj} is in {obj}.
- on: 9 templates; canonical = {subj} is on {obj}.
- left_of: 6 templates; canonical = {subj} is to the left of {obj}.
- right_of: 6 templates; canonical = {subj} is to the right of {obj}.
- above: 7 templates; canonical = {subj} is above {obj}.
- below: 7 templates; canonical = {subj} is below {obj}.
- near: 8 templates; canonical = {subj} is near {obj}.
- supports: 5 templates; canonical = {subj} supports {obj}.
- contains: 6 templates; canonical = {subj} contains {obj}.

Loaded relation priority from config:
{'on': 100, 'in': 100, 'above': 90, 'below': 90, 'left_of': 80, 'right_of': 80, 'near': 50, 'supports': 40, 'contains': 40}


In [ ]:
def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def stable_sort_relations(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return sorted(
        relations,
        key=lambda r: (
            r.get("scene", ""),
            r.get("chunk_id", ""),
            r.get("cluster_id", ""),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )


def stable_hash_int(text: str) -> int:
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest, 16)


print("IO helpers loaded.")

IO helpers loaded.


In [ ]:
def camel_to_snake(name: str) -> str:
    if not name:
        return "object"

    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1)
    return s2.lower()


def build_alias_map_from_relations(
    relations: List[Dict[str, Any]]
) -> Dict[str, Dict[str, Any]]:
    object_entries = {}

    for rel in relations:
        sid = rel["subject_id"]
        oid = rel["object_id"]

        if sid not in object_entries:
            object_entries[sid] = {
                "object_id": sid,
                "object_type": rel["subject_type"],
            }

        if oid not in object_entries:
            object_entries[oid] = {
                "object_id": oid,
                "object_type": rel["object_type"],
            }

    type_counter = defaultdict(int)
    alias_map = {}

    sorted_items = sorted(
        object_entries.items(),
        key=lambda item: (item[1]["object_type"], item[0])
    )

    for object_id, info in sorted_items:
        base = camel_to_snake(info["object_type"])
        type_counter[base] += 1

        object_uid = f"{base}_{type_counter[base]}"

        alias_map[object_id] = {
            "object_uid": object_uid,
            "object_id": object_id,
            "object_type": info["object_type"],
            "mention_text": object_uid,
        }

    return alias_map


def get_uid(alias_map: Dict[str, Dict[str, Any]], object_id: str) -> str:
    return alias_map[object_id]["object_uid"]


print("Alias helpers loaded.")

Alias helpers loaded.


In [ ]:
# Normalize relation labels.
#
# The inverse-relation maps come from pilot_full.config:
# INVERSE_TO_NATURAL and NATURAL_INVERSE_SWAP_RELATIONS.

def relation_priority_for_text(rel: Dict[str, Any]) -> int:
    return TEXT_RELATION_PRIORITY.get(rel.get("relation", ""), 0)


def normalize_relation_for_text(rel: Dict[str, Any]) -> Dict[str, Any]:
    r = rel["relation"]

    if ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT:
        return {
            "subject_id": rel["subject_id"],
            "subject_type": rel["subject_type"],
            "relation": r,
            "object_id": rel["object_id"],
            "object_type": rel["object_type"],
            "source_relation": rel,
            "was_swapped_for_text": False,
        }

    if r in NATURAL_INVERSE_SWAP_RELATIONS:
        return {
            "subject_id": rel["object_id"],
            "subject_type": rel["object_type"],
            "relation": INVERSE_TO_NATURAL[r],
            "object_id": rel["subject_id"],
            "object_type": rel["subject_type"],
            "source_relation": rel,
            "was_swapped_for_text": True,
        }

    return {
        "subject_id": rel["subject_id"],
        "subject_type": rel["subject_type"],
        "relation": r,
        "object_id": rel["object_id"],
        "object_type": rel["object_type"],
        "source_relation": rel,
        "was_swapped_for_text": False,
    }


def deduplicate_text_relations(text_relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    output = []

    for tr in text_relations:
        key = (tr["subject_id"], tr["relation"], tr["object_id"])

        if key in seen:
            continue

        seen.add(key)
        output.append(tr)

    return output


print("Relation normalization helpers loaded.")

Relation normalization helpers loaded.


In [ ]:
def select_template_for_relation(
    relation: str,
    mode: str,
    paragraph_id: str,
    relation_index: int,
    subject_uid: str,
    object_uid: str,
) -> Tuple[str, int]:
    templates = RELATION_TEMPLATES.get(
        relation,
        [f"{{subj}} is spatially related to {{obj}} by {relation}."]
    )

    if mode == CANONICAL_MODE:
        return templates[0], 0

    if mode == DIVERSE_MODE:
        # Use a deterministic random choice so reruns stay stable
        # while descriptions still vary across cases.
        key = f"{RANDOM_SEED}|{paragraph_id}|{relation_index}|{relation}|{subject_uid}|{object_uid}"
        idx = stable_hash_int(key) % len(templates)
        return templates[idx], idx

    raise ValueError(f"Unknown generation mode: {mode}")


def sentence_from_text_relation(
    text_rel: Dict[str, Any],
    alias_map: Dict[str, Dict[str, Any]],
    mode: str,
    paragraph_id: str,
    relation_index: int,
) -> Dict[str, Any]:
    s = get_uid(alias_map, text_rel["subject_id"])
    o = get_uid(alias_map, text_rel["object_id"])
    r = text_rel["relation"]

    template, template_index = select_template_for_relation(
        relation=r,
        mode=mode,
        paragraph_id=paragraph_id,
        relation_index=relation_index,
        subject_uid=s,
        object_uid=o,
    )

    sentence = template.format(subj=s, obj=o)

    return {
        "sentence": sentence,
        "template": template,
        "template_index": template_index,
        "generation_mode": mode,
    }


print("Template selection and sentence generation loaded.")

Template selection and sentence generation loaded.


In [ ]:
# Coverage-aware relation selection.
#
# This version controls the number of selected relations by relation type.
# It replaces the old filter_relations_for_text cell.

def get_relation_object_ids(rel: Dict[str, Any]) -> Tuple[str, str]:
    """
    Return the two object ids used by a relation.
    """
    return rel["subject_id"], rel["object_id"]


def get_cluster_object_ids(cluster: Dict[str, Any]) -> List[str]:
    """
    Get object ids from the cluster itself.

    Step 2 cluster usually preserves object_ids or objects.
    This helper is robust to both formats.
    """
    if "object_ids" in cluster and cluster["object_ids"]:
        return list(cluster["object_ids"])

    if "objects" in cluster and cluster["objects"]:
        return [
            obj["objectId"]
            for obj in cluster["objects"]
            if "objectId" in obj
        ]

    # Fallback: infer from relations
    ids = set()
    for rel in cluster.get("relations", []):
        ids.add(rel["subject_id"])
        ids.add(rel["object_id"])

    return sorted(ids)


def relation_key(rel: Dict[str, Any]) -> Tuple[str, str, str]:
    """
    Unique directed relation key.
    """
    return (
        rel["subject_id"],
        rel["relation"],
        rel["object_id"],
    )


def relation_base_score(rel: Dict[str, Any]) -> int:

    """
    Return the base relation score before the object-coverage reward is added.
    In the current full-pipeline output, Step 2 uses relation_family values
    such as "topological" and "geometric". Older labels such as "support",
    "containment", and "vertical" are not used here because they are no longer
    reliable for this version of the pipeline.
    """
    score = relation_priority_for_text(rel)

    if rel.get("preferred_for_paragraph", False):
        score += 40

    # Prefer stronger spatial/topological relations over weak geometric "near".
    if rel.get("relation_family") == "topological":
        score += 20

    if rel.get("relation") in {"in", "contains", "on", "supports", "above", "below"}:
        score += 10

    if rel.get("relation") == "near":
        score -= 20

    return score


def relation_variant_jitter(
    rel: Dict[str, Any],
    cluster: Dict[str, Any],
    paragraph_index: int,
) -> int:
    """
    Stable pseudo-random jitter.

    Different paragraph_index leads to slightly different relation selection,
    while keeping the result reproducible under RANDOM_SEED.

    paragraph_index == 0 keeps the original deterministic ranking.
    """
    if paragraph_index == 0:
        return 0

    cluster_identity = (
        cluster.get("cluster_id")
        or cluster.get("chunk_id")
        or "|".join(get_cluster_object_ids(cluster))
    )

    key = (
        f"{RANDOM_SEED}|"
        f"{cluster_identity}|"
        f"{paragraph_index}|"
        f"{rel['subject_id']}|"
        f"{rel['relation']}|"
        f"{rel['object_id']}"
    )

    span = 2 * RELATION_SELECTION_JITTER + 1
    return stable_hash_int(key) % span - RELATION_SELECTION_JITTER


def filter_candidate_relations_for_text(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    First-stage filtering: remove relations that should not be verbalized.
    """
    candidates = []

    for rel in relations:
        if not rel.get("verbalizable", True):
            continue

        if not rel.get("candidate_for_text", True):
            continue

        if rel.get("relation") == "near" and not ALLOW_NEAR_IN_TEXT:
            continue

        candidates.append(rel)

    # Prefer Step 2 preferred relations without losing objects that only appear
    # in non-preferred relations.
    if USE_PREFERRED_RELATIONS_FIRST:
        preferred = [
            rel for rel in candidates
            if rel.get("preferred_for_paragraph", False)
        ]

        if len(preferred) >= MIN_RELATIONS_PER_PARAGRAPH:
            preferred_keys = {
                relation_key(r)
                for r in preferred
            }

            non_preferred = [
                r for r in candidates
                if relation_key(r) not in preferred_keys
            ]

            candidates = preferred + non_preferred

    return candidates


def selection_score_for_relation(
    rel: Dict[str, Any],
    cluster: Dict[str, Any],
    paragraph_index: int,
    avoid_relation_keys: set,
) -> int:
    """
    Relation quality score with paragraph-specific variation
    and repeated-relation penalty.
    """
    key = relation_key(rel)

    score = relation_base_score(rel)

    if key in avoid_relation_keys:
        score -= REUSED_RELATION_PENALTY

    score += relation_variant_jitter(
        rel=rel,
        cluster=cluster,
        paragraph_index=paragraph_index,
    )

    return score


def select_relations_for_object_coverage(
    relations: List[Dict[str, Any]],
    cluster: Dict[str, Any],
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> List[Dict[str, Any]]:
"""
Select relations for paragraph generation.

The selection aims to provide enough explicit relation sentences for probing
while still covering the paragraph objects reasonably well. It continues until
TARGET_RELATIONS_PER_PARAGRAPH is reached, unless the maximum relation count or
other constraints stop it earlier.
"""
    if avoid_relation_keys is None:
        avoid_relation_keys = set()

    candidates = filter_candidate_relations_for_text(relations)

    if not candidates:
        return []

    cluster_object_ids = get_cluster_object_ids(cluster)
    cluster_object_id_set = set(cluster_object_ids)

    target_num_objects = max(
        MIN_RELATIONS_PER_PARAGRAPH,
        int(round(len(cluster_object_id_set) * TARGET_OBJECT_COVERAGE_RATIO))
    )

    if MAX_OBJECTS_PER_PARAGRAPH is not None:
        target_num_objects = min(target_num_objects, MAX_OBJECTS_PER_PARAGRAPH)

    target_num_relations = min(
        TARGET_RELATIONS_PER_PARAGRAPH,
        MAX_RELATIONS_PER_PARAGRAPH,
    )

    selected = []
    selected_keys = set()
    covered_objects = set()
    near_count = 0
    redundant_count = 0

    remaining = sorted(
        candidates,
        key=lambda r: (
            -selection_score_for_relation(
                rel=r,
                cluster=cluster,
                paragraph_index=paragraph_index,
                avoid_relation_keys=avoid_relation_keys,
            ),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )

    while remaining and len(selected) < MAX_RELATIONS_PER_PARAGRAPH:
        best_idx = None
        best_score = None

        for idx, rel in enumerate(remaining):
            s, o = get_relation_object_ids(rel)
            key = relation_key(rel)

            if key in selected_keys:
                continue

            if rel.get("relation") == "near" and near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue

            rel_objects = {s, o}
            new_objects = rel_objects - covered_objects
            num_new = len(new_objects)

            if MAX_OBJECTS_PER_PARAGRAPH is not None:
                possible_object_count = len(covered_objects | rel_objects)
                if possible_object_count > MAX_OBJECTS_PER_PARAGRAPH:
                    continue

            if num_new == 0:
                if not ALLOW_REDUNDANT_RELATIONS:
                    continue

                if redundant_count >= MAX_REDUNDANT_RELATIONS:
                    continue

            score = selection_score_for_relation(
                rel=rel,
                cluster=cluster,
                paragraph_index=paragraph_index,
                avoid_relation_keys=avoid_relation_keys,
            )

            if len(covered_objects) < target_num_objects:
                if num_new == 1:
                    score += NEW_OBJECT_BONUS
                elif num_new == 2:
                    score += NEW_OBJECT_BONUS + TWO_NEW_OBJECT_BONUS
            else:
                # Once coverage is reached, add relations among already mentioned
                # objects before introducing more object mentions.
                if num_new == 0:
                    score += 20
                elif num_new == 1:
                    score += 5
                elif num_new == 2:
                    score += 0

            if s in cluster_object_id_set:
                score += 5
            if o in cluster_object_id_set:
                score += 5

            tie_break = (
                score,
                relation_base_score(rel),
                num_new,
                -idx,
            )

            if best_score is None or tie_break > best_score:
                best_score = tie_break
                best_idx = idx

        if best_idx is None:
            break

        chosen = remaining.pop(best_idx)
        s, o = get_relation_object_ids(chosen)
        key = relation_key(chosen)

        before = set(covered_objects)

        selected.append(chosen)
        selected_keys.add(key)
        covered_objects.update([s, o])

        if chosen.get("relation") == "near":
            near_count += 1

        if len(covered_objects - before) == 0:
            redundant_count += 1

        # Object coverage alone is not enough to stop selection; the target
        # relation count must also be reached.
        coverage_satisfied = len(covered_objects) >= target_num_objects
        target_relation_count_satisfied = len(selected) >= target_num_relations

        if coverage_satisfied and target_relation_count_satisfied:
            break

        if len(selected) >= MAX_RELATIONS_PER_PARAGRAPH:
            break

    # If the relation target is still not met, add the best remaining relations.
    if len(selected) < MIN_RELATIONS_PER_PARAGRAPH:
        selected_keys = {
            relation_key(r)
            for r in selected
        }

        fallback = sorted(
            candidates,
            key=lambda r: (
                -selection_score_for_relation(
                    rel=r,
                    cluster=cluster,
                    paragraph_index=paragraph_index,
                    avoid_relation_keys=avoid_relation_keys,
                ),
                r.get("relation", ""),
                r.get("subject_type", ""),
                r.get("object_type", ""),
                r.get("subject_id", ""),
                r.get("object_id", ""),
            )
        )

        for rel in fallback:
            if len(selected) >= MIN_RELATIONS_PER_PARAGRAPH:
                break

            key = relation_key(rel)

            if key in selected_keys:
                continue

            if rel.get("relation") == "near" and near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue

            s, o = get_relation_object_ids(rel)

            if MAX_OBJECTS_PER_PARAGRAPH is not None:
                possible_object_count = len(covered_objects | {s, o})
                if possible_object_count > MAX_OBJECTS_PER_PARAGRAPH:
                    continue

            selected.append(rel)
            selected_keys.add(key)
            covered_objects.update([s, o])

            if rel.get("relation") == "near":
                near_count += 1

    return selected


def filter_relations_for_text(
    relations: List[Dict[str, Any]],
    cluster: Optional[Dict[str, Any]] = None,
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> List[Dict[str, Any]]:
    """
    Choose the relation selector used for paragraph generation.
    Cluster-level selection uses coverage-aware sampling with paragraph-level
    variation. Calls without a cluster use the priority-based fallback.
    """
    if cluster is not None:
        return select_relations_for_object_coverage(
            relations=relations,
            cluster=cluster,
            paragraph_index=paragraph_index,
            avoid_relation_keys=avoid_relation_keys,
        )

    candidates = filter_candidate_relations_for_text(relations)

    candidates = sorted(
        candidates,
        key=lambda r: (
            -relation_priority_for_text(r),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )

    selected = []
    near_count = 0

    for rel in candidates:
        if len(selected) >= MAX_RELATIONS_PER_PARAGRAPH:
            break

        if rel.get("relation") == "near":
            if near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue
            near_count += 1

        selected.append(rel)

    return selected


print("Coverage-aware relation selection helpers loaded.")

Coverage-aware relation selection helpers loaded.


In [ ]:
# Build pair-level targets.
#
# The inverse label map comes from pilot_full.config.

def build_relation_label_index(
    relations: List[Dict[str, Any]]
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    index = {}

    for rel in stable_sort_relations(relations):
        s = rel["subject_id"]
        o = rel["object_id"]
        r = rel["relation"]

        key = (s, o)

        if key not in index:
            index[key] = {
                "relation_label": r,
                "relation_record": rel,
                "relation_source": "direct",
            }

    current_items = list(index.items())

    for (s, o), info in current_items:
        r = info["relation_label"]

        if r not in INVERSE_LABEL_MAP:
            continue

        inverse_r = INVERSE_LABEL_MAP[r]
        reverse_key = (o, s)

        if reverse_key not in index:
            index[reverse_key] = {
                "relation_label": inverse_r,
                "relation_record": info["relation_record"],
                "relation_source": "inferred_inverse",
            }

    return index


def build_pair_targets_for_paragraph(
    paragraph_id: str,
    scene: str,
    chunk_id: str,
    cluster_id: str,
    alias_map: Dict[str, Dict[str, Any]],
    all_cluster_relations: List[Dict[str, Any]],
    explicit_text_relations: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    object_ids = sorted(alias_map.keys())
    relation_index = build_relation_label_index(all_cluster_relations)

    explicit_unordered_pairs = set()
    explicit_directed_pairs = set()

    for tr in explicit_text_relations:
        s = tr["subject_id"]
        o = tr["object_id"]

        explicit_unordered_pairs.add(tuple(sorted([s, o])))
        explicit_directed_pairs.add((s, o))

    pair_targets = []

    for subject_id in object_ids:
        for object_id in object_ids:
            if subject_id == object_id:
                continue

            key = (subject_id, object_id)
            info = relation_index.get(key)

            if info is None:
                if not INCLUDE_NONE_RELATION_PAIRS:
                    continue

                relation_label = NONE_RELATION_LABEL
                relation_source = "none"
                has_relation_label = False
                geometry_label = None
                source_relation_record = None

            else:
                relation_label = info["relation_label"]
                relation_source = info["relation_source"]
                has_relation_label = True
                source_relation_record = info["relation_record"]
                geometry_label = source_relation_record.get("geometry_label")

            unordered = tuple(sorted([subject_id, object_id]))

            if key in explicit_directed_pairs:
                pair_evidence_type = "explicit_direct"
            elif unordered in explicit_unordered_pairs:
                pair_evidence_type = "explicit_inverse_or_same_sentence_pair"
            elif has_relation_label:
                pair_evidence_type = "implicit_labeled"
            else:
                pair_evidence_type = "implicit_none"

            s_info = alias_map[subject_id]
            o_info = alias_map[object_id]

            pair_targets.append({
                "paragraph_id": paragraph_id,
                "scene": scene,
                "chunk_id": chunk_id,
                "cluster_id": cluster_id,

                "subject_uid": s_info["object_uid"],
                "subject_id": subject_id,
                "subject_type": s_info["object_type"],

                "object_uid": o_info["object_uid"],
                "object_id": object_id,
                "object_type": o_info["object_type"],

                "relation_label": relation_label,
                "has_relation_label": has_relation_label,
                "relation_source": relation_source,
                "pair_evidence_type": pair_evidence_type,

                "geometry_label": geometry_label,

                "source_relation_summary": None if source_relation_record is None else {
                    "subject_id": source_relation_record.get("subject_id"),
                    "relation": source_relation_record.get("relation"),
                    "object_id": source_relation_record.get("object_id"),
                    "relation_family": source_relation_record.get("relation_family"),
                }
            })

    return pair_targets


print("Pair target helpers loaded.")

Pair target helpers loaded.


In [ ]:
# Build a paragraph record from one cluster.
#
# Relations are selected with the coverage-aware selector.
# Intro templates come from pilot_full.config.

def make_paragraph_intro(
    scene: str,
    scene_type: str,
    chunk_id: str,
    cluster_id: str,
    mode: str,
    paragraph_index: int = 0,
) -> str:
    """
    Select a scene-type-aware paragraph opening.

    Prefer cfg.INTRO_TEMPLATES_BY_SCENE_TYPE when it exists. If the uploaded
    config still only has the old kitchen-specific INTRO_TEMPLATES, use a safe
    local fallback so living room / bedroom / bathroom scenes are not described
    as kitchens.
    """
    fallback_templates_by_type = {
        "kitchen": [
            "In this local kitchen area,",
            "In this part of the kitchen,",
            "Within this kitchen scene,",
            "In the described kitchen area,",
        ],
        "living_room": [
            "In this local living room area,",
            "In this part of the living room,",
            "Within this living room scene,",
            "In the described living room area,",
        ],
        "bedroom": [
            "In this local bedroom area,",
            "In this part of the bedroom,",
            "Within this bedroom scene,",
            "In the described bedroom area,",
        ],
        "bathroom": [
            "In this local bathroom area,",
            "In this part of the bathroom,",
            "Within this bathroom scene,",
            "In the described bathroom area,",
        ],
        "unknown": [
            "In this local scene area,",
            "In this part of the scene,",
            "Within this local scene,",
            "In the described scene area,",
        ],
    }

    templates_by_type = getattr(
        cfg,
        "INTRO_TEMPLATES_BY_SCENE_TYPE",
        fallback_templates_by_type,
    )

    templates = templates_by_type.get(
        scene_type,
        templates_by_type.get("unknown"),
    )

    if not templates:
        templates = fallback_templates_by_type["unknown"]

    key = f"{RANDOM_SEED}|{mode}|{scene}|{scene_type}|{chunk_id}|{cluster_id}|{paragraph_index}"
    idx = stable_hash_int(key) % len(templates)
    return templates[idx]


def build_paragraph_record_from_cluster(
    scene: str,
    scene_type: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    mode: str,
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> Optional[Dict[str, Any]]:
    cluster_relations = cluster.get("relations", []) or []

    if not cluster_relations:
        return None

    if avoid_relation_keys is None:
        avoid_relation_keys = set()

    selected_relations = filter_relations_for_text(
        cluster_relations,
        cluster=cluster,
        paragraph_index=paragraph_index,
        avoid_relation_keys=avoid_relation_keys,
    )

    if len(selected_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        return None

    text_relations = [
        normalize_relation_for_text(rel)
        for rel in selected_relations
    ]
    text_relations = deduplicate_text_relations(text_relations)

    if len(text_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        return None

    alias_map = build_alias_map_from_relations(text_relations)

    if MAX_OBJECTS_PER_PARAGRAPH is not None:
        if len(alias_map) > MAX_OBJECTS_PER_PARAGRAPH:
            return None

    paragraph_id = f"{scene}_{cluster['cluster_id']}_step3_{mode}_{paragraph_index}"

    sentence_records = []
    sentences = []

    for relation_index, tr in enumerate(text_relations):
        sent_info = sentence_from_text_relation(
            text_rel=tr,
            alias_map=alias_map,
            mode=mode,
            paragraph_id=paragraph_id,
            relation_index=relation_index,
        )

        sentences.append(sent_info["sentence"])

        sentence_records.append({
            "sentence": sent_info["sentence"],
            "template": sent_info["template"],
            "template_index": sent_info["template_index"],
            "generation_mode": mode,

            "subject_uid": get_uid(alias_map, tr["subject_id"]),
            "subject_id": tr["subject_id"],
            "subject_type": tr["subject_type"],

            "relation": tr["relation"],

            "object_uid": get_uid(alias_map, tr["object_id"]),
            "object_id": tr["object_id"],
            "object_type": tr["object_type"],

            "was_swapped_for_text": tr["was_swapped_for_text"],
            "source_relation": {
                "subject_id": tr["source_relation"].get("subject_id"),
                "relation": tr["source_relation"].get("relation"),
                "object_id": tr["source_relation"].get("object_id"),
                "relation_family": tr["source_relation"].get("relation_family"),
            },
        })

    intro = make_paragraph_intro(
        scene=scene,
        scene_type=scene_type,
        chunk_id=chunk["chunk_id"],
        cluster_id=cluster["cluster_id"],
        mode=mode,
        paragraph_index=paragraph_index,
    )

    text = intro + " " + " ".join(sentences)

    object_mentions = list(alias_map.values())
    object_mentions = sorted(object_mentions, key=lambda x: x["object_uid"])

    pair_targets = build_pair_targets_for_paragraph(
        paragraph_id=paragraph_id,
        scene=scene,
        chunk_id=chunk["chunk_id"],
        cluster_id=cluster["cluster_id"],
        alias_map=alias_map,
        all_cluster_relations=cluster_relations,
        explicit_text_relations=text_relations,
    )

    label_counts = dict(Counter(pt["relation_label"] for pt in pair_targets))
    evidence_counts = dict(Counter(pt["pair_evidence_type"] for pt in pair_targets))

    source_cluster_object_ids = get_cluster_object_ids(cluster)
    source_cluster_num_objects = len(source_cluster_object_ids)
    num_object_mentions = len(object_mentions)

    if source_cluster_num_objects > 0:
        object_coverage_ratio = num_object_mentions / source_cluster_num_objects
    else:
        object_coverage_ratio = None

    selected_relation_keys = [
        relation_key(rel)
        for rel in selected_relations
    ]

    text_relation_keys = [
        (
            tr["subject_id"],
            tr["relation"],
            tr["object_id"],
        )
        for tr in text_relations
    ]

    return {
        "paragraph_id": paragraph_id,
        "experiment": EXPERIMENT_NAME,
        "generation_mode": mode,

        "scene": scene,
        "scene_type": scene_type,
        "chunk_id": chunk["chunk_id"],
        "cluster_id": cluster["cluster_id"],
        "source_chunk_id": cluster.get("source_chunk_id", chunk["chunk_id"]),
        "grid_i": cluster.get("grid_i"),
        "grid_j": cluster.get("grid_j"),

        "paragraph_index_within_cluster": paragraph_index,

        "text": text,

        "object_mentions": object_mentions,
        "explicit_relations_in_text": sentence_records,
        "pair_targets": pair_targets,

        "diagnostics": {
            "num_text_relations": len(text_relations),
            "target_relations_per_paragraph": TARGET_RELATIONS_PER_PARAGRAPH,
            "max_relations_per_paragraph": MAX_RELATIONS_PER_PARAGRAPH,
            "min_relations_per_paragraph": MIN_RELATIONS_PER_PARAGRAPH,

            "num_object_mentions": num_object_mentions,
            "num_pair_targets": len(pair_targets),

            "source_cluster_num_objects": source_cluster_num_objects,
            "target_object_coverage_ratio": TARGET_OBJECT_COVERAGE_RATIO,
            "object_coverage_ratio": object_coverage_ratio,
            "max_objects_per_paragraph": MAX_OBJECTS_PER_PARAGRAPH,

            "paragraph_index_within_cluster": paragraph_index,
            "num_avoid_relation_keys_received": len(avoid_relation_keys),
            "selected_relation_keys": selected_relation_keys,
            "text_relation_keys": text_relation_keys,

            "pair_label_counts": label_counts,
            "pair_evidence_counts": evidence_counts,
            "text_length_chars": len(text),
            "text_length_words_approx": len(text.split()),
            "source_cluster_num_relations": cluster.get("num_relations"),
            "source_cluster_relation_counts": cluster.get("relation_counts"),
            "source_cluster_preferred_relation_counts": cluster.get("preferred_relation_counts"),
        }
    }


print("Coverage-aware cluster paragraph builder loaded.")

Coverage-aware cluster paragraph builder loaded.


In [ ]:
# Build the Step 3 output for one scene.

def build_scene_step3(scene_data: Dict[str, Any], mode: str) -> Dict[str, Any]:
    """
    Build Step 3 output for one scene under one selected generation mode.

    mode:
        "canonical" or "diverse"
    """
    scene = scene_data["scene"]
    scene_type = scene_data.get("scene_type")

    if scene_type is None:
        scene_type = cfg.infer_scene_type_from_floorplan(scene)

    paragraph_records = []

    for chunk in scene_data.get("chunks", []):
        for cluster in chunk.get("clusters", []):

            # Avoid reusing the same explicit relations across paragraphs from the same cluster.
            used_relation_keys_for_cluster = set()

            for paragraph_index in range(MAX_PARAGRAPHS_PER_CLUSTER):
                record = build_paragraph_record_from_cluster(
                    scene=scene,
                    scene_type=scene_type,
                    chunk=chunk,
                    cluster=cluster,
                    mode=mode,
                    paragraph_index=paragraph_index,
                    avoid_relation_keys=used_relation_keys_for_cluster,
                )

                if record is not None:
                    paragraph_records.append(record)

                    # Mark text-facing explicit relations as used.
                    for rel in record["explicit_relations_in_text"]:
                        used_relation_keys_for_cluster.add(
                            (
                                rel["subject_id"],
                                rel["relation"],
                                rel["object_id"],
                            )
                        )

                    # Keep the source key as well, since normalization may map
                    # labels such as supports -> on or contains -> in.
                    for rel in record["explicit_relations_in_text"]:
                        src = rel.get("source_relation", {})
                        if src.get("subject_id") and src.get("relation") and src.get("object_id"):
                            used_relation_keys_for_cluster.add(
                                (
                                    src["subject_id"],
                                    src["relation"],
                                    src["object_id"],
                                )
                            )

    all_pair_targets = []
    for rec in paragraph_records:
        all_pair_targets.extend(rec["pair_targets"])

    paragraph_relation_counts = dict(
        Counter(
            rel["relation"]
            for rec in paragraph_records
            for rel in rec["explicit_relations_in_text"]
        )
    )

    pair_label_counts = dict(
        Counter(
            pt["relation_label"]
            for pt in all_pair_targets
        )
    )

    pair_evidence_counts = dict(
        Counter(
            pt["pair_evidence_type"]
            for pt in all_pair_targets
        )
    )

    paragraph_text_relation_counts = [
        rec["diagnostics"]["num_text_relations"]
        for rec in paragraph_records
    ]

    return {
        "scene": scene,
        "scene_type": scene_type,
        "experiment": EXPERIMENT_NAME,
        "generation_mode": mode,

        "source_step2_summary": {
            "scene_type": scene_type,
            "num_chunks": scene_data.get("num_chunks"),
            "num_relations": scene_data.get("num_relations"),
            "relation_counts": scene_data.get("relation_counts"),
            "preferred_relation_counts": scene_data.get("preferred_relation_counts"),
            "num_geometry_pairs": scene_data.get("num_geometry_pairs"),
        },

        "step3_config": {
            "generation_mode": mode,
            "max_paragraphs_per_cluster": MAX_PARAGRAPHS_PER_CLUSTER,
            "max_relations_per_paragraph": MAX_RELATIONS_PER_PARAGRAPH,
            "min_relations_per_paragraph": MIN_RELATIONS_PER_PARAGRAPH,
            "target_relations_per_paragraph": TARGET_RELATIONS_PER_PARAGRAPH,
            "use_preferred_relations_first": USE_PREFERRED_RELATIONS_FIRST,
            "allow_near_in_text": ALLOW_NEAR_IN_TEXT,
            "max_near_sentences_per_paragraph": MAX_NEAR_SENTENCES_PER_PARAGRAPH,
            "allow_inverse_topological_in_text": ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT,
            "include_none_relation_pairs": INCLUDE_NONE_RELATION_PAIRS,
            "none_relation_label": NONE_RELATION_LABEL,

            "target_object_coverage_ratio": TARGET_OBJECT_COVERAGE_RATIO,
            "max_objects_per_paragraph": MAX_OBJECTS_PER_PARAGRAPH,
            "new_object_bonus": NEW_OBJECT_BONUS,
            "two_new_object_bonus": TWO_NEW_OBJECT_BONUS,
            "allow_redundant_relations": ALLOW_REDUNDANT_RELATIONS,
            "max_redundant_relations": MAX_REDUNDANT_RELATIONS,

            "relation_selection_jitter": RELATION_SELECTION_JITTER,
            "reused_relation_penalty": REUSED_RELATION_PENALTY,

            "random_seed": RANDOM_SEED,
        },

        "num_paragraphs": len(paragraph_records),
        "num_pair_targets": len(all_pair_targets),

        "paragraph_relation_counts": paragraph_relation_counts,
        "pair_label_counts": pair_label_counts,
        "pair_evidence_counts": pair_evidence_counts,

        "paragraph_text_relation_count_summary": {
            "min": min(paragraph_text_relation_counts) if paragraph_text_relation_counts else None,
            "max": max(paragraph_text_relation_counts) if paragraph_text_relation_counts else None,
            "mean": (
                sum(paragraph_text_relation_counts) / len(paragraph_text_relation_counts)
                if paragraph_text_relation_counts else None
            ),
            "counts": paragraph_text_relation_counts,
        },

        "paragraphs": paragraph_records,
    }


print("Scene-level Step 3 builder loaded.")

Scene-level Step 3 builder loaded.


In [ ]:
# Run Step 3 with the configured generation mode.
#
# RUN_MODE comes from pilot_full.config. All available Step 2 files are
# processed automatically.

def natural_sort_key(filename):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", filename)
    ]


if RUN_MODE not in {CANONICAL_MODE, DIVERSE_MODE}:
    raise ValueError(
        f"Invalid RUN_MODE: {RUN_MODE}. "
        f"Use '{CANONICAL_MODE}' or '{DIVERSE_MODE}'."
    )

if RUN_MODE == CANONICAL_MODE:
    OUTPUT_DIR = STEP3_CANONICAL_OUTPUT_DIR
elif RUN_MODE == DIVERSE_MODE:
    OUTPUT_DIR = STEP3_DIVERSE_OUTPUT_DIR
else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Selected RUN_MODE:", RUN_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)

expected_input_files = sorted(
    [
        filename
        for filename in os.listdir(STEP3_INPUT_DIR)
        if filename.startswith("FloorPlan") and filename.endswith("_chunk_relations.json")
    ],
    key=natural_sort_key
)

if not expected_input_files:
    raise FileNotFoundError(
        f"No Step 2 relation files found in directory:\n{STEP3_INPUT_DIR}\n\n"
        "Expected files ending with '_chunk_relations.json'."
    )

SCENES = [
    filename.replace("_chunk_relations.json", "")
    for filename in expected_input_files
]

expected_scene_set = set(cfg.SCENES)
found_scene_set = set(SCENES)

missing_scenes = sorted(
    expected_scene_set - found_scene_set,
    key=natural_sort_key,
)

extra_scenes = sorted(
    found_scene_set - expected_scene_set,
    key=natural_sort_key,
)

print("Detected scenes:", SCENES)
print("Number of input files:", len(expected_input_files))

print("\nScene coverage check:")
print(f"Configured scenes in cfg.SCENES: {len(expected_scene_set)}")
print(f"Available Step 2 relation files: {len(found_scene_set)}")
print(f"Missing from configured scene list: {len(missing_scenes)}")
print(f"Extra files not listed in cfg.SCENES: {len(extra_scenes)}")

if missing_scenes:
    print("\n[Info] The following configured scenes do not have Step 2 relation files:")
    for scene in missing_scenes:
        print("-", scene)

if extra_scenes:
    print("\n[Warning] The following Step 2 files are not listed in cfg.SCENES:")
    for scene in extra_scenes:
        print("-", scene)

print("\nStep 3 will process all available Step 2 relation files listed in SCENES.")

def run_step3_for_mode(mode: str, output_dir: str):
    processed = []

    print(f"\n=== Running Step 3 mode: {mode} ===")

    for filename in expected_input_files:
        in_path = os.path.join(STEP3_INPUT_DIR, filename)
        scene_data = load_json(in_path)

        out_data = build_scene_step3(scene_data, mode=mode)

        out_filename = f"{out_data['scene']}_step3_text_{mode}.json"
        out_path = os.path.join(output_dir, out_filename)

        save_json(out_path, out_data)
        processed.append(out_filename)

        print(
            f"[Saved] {out_filename} | "
            f"scene={out_data['scene']} | "
            f"scene_type={out_data.get('scene_type')} | "
            f"paragraphs={out_data['num_paragraphs']} | "
            f"pair_targets={out_data['num_pair_targets']} | "
            f"paragraph_relation_counts={out_data['paragraph_relation_counts']}"
        )

    return processed


processed_files = run_step3_for_mode(
    mode=RUN_MODE,
    output_dir=OUTPUT_DIR,
)

print(f"\nOutput files for mode={RUN_MODE}:")
for f in processed_files:
    print("-", f)

Selected RUN_MODE: diverse
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div
Detected scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6', 'FloorPlan7', 'FloorPlan8', 'FloorPlan9', 'FloorPlan10', 'FloorPlan11', 'FloorPlan12', 'FloorPlan13', 'FloorPlan14', 'FloorPlan15', 'FloorPlan16', 'FloorPlan17', 'FloorPlan18', 'FloorPlan19', 'FloorPlan20', 'FloorPlan21', 'FloorPlan22', 'FloorPlan23', 'FloorPlan24', 'FloorPlan25', 'FloorPlan26', 'FloorPlan27', 'FloorPlan28', 'FloorPlan29', 'FloorPlan30', 'FloorPlan201', 'FloorPlan202', 'FloorPlan203', 'FloorPlan204', 'FloorPlan205', 'FloorPlan206', 'FloorPlan207', 'FloorPlan208', 'FloorPlan209', 'FloorPlan210', 'FloorPlan211', 'FloorPlan212', 'FloorPlan213', 'FloorPlan214', 'FloorPlan215', 'FloorPlan216', 'FloorPlan217', 'FloorPlan218', 'FloorPlan219', 'FloorPlan220', 'FloorPlan221', 'FloorPlan222', 'FloorPlan223', 'FloorPlan224', 'FloorPlan225', 'FloorPla

In [ ]:
def iter_scene_output_files(output_dir: str, mode: str):
    for scene in SCENES:
        filename = f"{scene}_step3_text_{mode}.json"
        path = os.path.join(output_dir, filename)

        if not os.path.exists(path):
            print("[Skip missing output]", filename)
            continue

        yield scene, filename, path


def export_jsonl(output_dir: str, mode: str) -> str:
    """
    Export paragraph-level records.
    Each line is one paragraph record, including text, object_mentions,
    explicit_relations_in_text, and pair_targets.
    """
    jsonl_path = os.path.join(output_dir, f"step3_paragraphs_{mode}_all.jsonl")

    num_written = 0

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for scene, filename, path in iter_scene_output_files(output_dir, mode):
            data = load_json(path)

            for rec in data["paragraphs"]:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                num_written += 1

    print(f"[Paragraph JSONL] mode={mode} | path={jsonl_path} | paragraphs={num_written}")
    return jsonl_path


def export_probe_pairs_jsonl(output_dir: str, mode: str) -> str:
    """
    Export pair-level records for quick inspection.

    This is not the final Step 4 probe dataset.
    Step 4 remains responsible for the final clean direct/inverse probe export.
    """
    out_path = os.path.join(
        output_dir,
        f"step3_probe_pairs_{mode}_labeled_only.jsonl"
    )

    keep_evidence_types = {
        "explicit_direct",
        "explicit_inverse_or_same_sentence_pair",
        "implicit_labeled",
    }

    num_written = 0
    label_counts = Counter()
    evidence_counts = Counter()
    paragraph_counts = Counter()

    with open(out_path, "w", encoding="utf-8") as wf:
        for scene, filename, path in iter_scene_output_files(output_dir, mode):
            data = load_json(path)

            for rec in data["paragraphs"]:
                for pt in rec["pair_targets"]:
                    if pt["pair_evidence_type"] not in keep_evidence_types:
                        continue

                    if pt["relation_label"] == NONE_RELATION_LABEL:
                        continue

                    row = {
                        "paragraph_id": rec["paragraph_id"],
                        "scene": rec["scene"],
                        "scene_type": rec.get("scene_type"),
                        "chunk_id": rec["chunk_id"],
                        "cluster_id": rec["cluster_id"],
                        "generation_mode": rec["generation_mode"],
                        "paragraph_index_within_cluster": rec.get("paragraph_index_within_cluster"),

                        "text": rec["text"],

                        "subject_uid": pt["subject_uid"],
                        "subject_id": pt["subject_id"],
                        "subject_type": pt["subject_type"],

                        "object_uid": pt["object_uid"],
                        "object_id": pt["object_id"],
                        "object_type": pt["object_type"],

                        "relation_label": pt["relation_label"],
                        "has_relation_label": pt["has_relation_label"],
                        "relation_source": pt["relation_source"],
                        "pair_evidence_type": pt["pair_evidence_type"],
                        "geometry_label": pt.get("geometry_label"),

                        "source_relation_summary": pt.get("source_relation_summary"),
                    }

                    wf.write(json.dumps(row, ensure_ascii=False) + "\n")

                    num_written += 1
                    label_counts[pt["relation_label"]] += 1
                    evidence_counts[pt["pair_evidence_type"]] += 1
                    paragraph_counts[rec["paragraph_id"]] += 1

    print(f"[Probe-pair JSONL] mode={mode} | path={out_path}")
    print("num_probe_pairs:", num_written)
    print("num_paragraphs_with_probe_pairs:", len(paragraph_counts))
    print("label_counts:", dict(label_counts))
    print("evidence_counts:", dict(evidence_counts))

    return out_path


jsonl_path = export_jsonl(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

probe_jsonl_path = export_probe_pairs_jsonl(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

[Paragraph JSONL] mode=diverse | path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div/step3_paragraphs_diverse_all.jsonl | paragraphs=1932
[Probe-pair JSONL] mode=diverse | path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step3_div/step3_probe_pairs_diverse_labeled_only.jsonl
num_probe_pairs: 94230
num_paragraphs_with_probe_pairs: 1932
label_counts: {'right_of': 22530, 'above': 6276, 'contains': 1076, 'near': 16774, 'on': 8846, 'supports': 8840, 'below': 6276, 'in': 1082, 'left_of': 22530}
evidence_counts: {'implicit_labeled': 46414, 'explicit_direct': 24252, 'explicit_inverse_or_same_sentence_pair': 23564}


In [ ]:
OBJECT_ID_LEAK_PATTERN = re.compile(r"[A-Za-z]+?\|[-+0-9\.]+\|[-+0-9\.]+\|[-+0-9\.]+")
ALIAS_PATTERN = re.compile(r"\b[a-z]+(?:_[a-z]+)*_\d+\b")


def split_sentences_simple(text: str):
    parts = [s.strip() for s in text.split(".")]
    return [s for s in parts if s]


def check_paragraph_content(rec):
    issues = []

    text = rec.get("text", "")
    object_mentions = rec.get("object_mentions", [])
    explicit_relations = rec.get("explicit_relations_in_text", [])
    pair_targets = rec.get("pair_targets", [])

    aliases_from_metadata = {
        obj["object_uid"]
        for obj in object_mentions
    }

    aliases_in_text = set(ALIAS_PATTERN.findall(text))
    sentences = split_sentences_simple(text)

    if not text.strip():
        issues.append("empty_text")

    if len(sentences) < MIN_RELATIONS_PER_PARAGRAPH:
        issues.append(f"too_few_sentences: {len(sentences)}")

    if len(text.split()) > 220:
        issues.append(f"text_too_long_words: {len(text.split())}")

    if OBJECT_ID_LEAK_PATTERN.search(text):
        issues.append("object_id_or_coordinate_leakage")

    missing_aliases = sorted(aliases_from_metadata - aliases_in_text)
    if missing_aliases:
        issues.append(f"metadata_aliases_missing_in_text: {missing_aliases}")

    extra_aliases = sorted(aliases_in_text - aliases_from_metadata)
    if extra_aliases:
        issues.append(f"unregistered_aliases_in_text: {extra_aliases}")

    sentence_counts = Counter(sentences)
    duplicate_sentences = [
        sent for sent, count in sentence_counts.items()
        if count > 1
    ]

    if duplicate_sentences:
        issues.append(f"duplicate_sentences: {duplicate_sentences[:3]}")

    if len(explicit_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        issues.append(f"too_few_explicit_relations: {len(explicit_relations)}")

    n = len(object_mentions)
    expected_ordered_pairs = n * (n - 1)

    if INCLUDE_NONE_RELATION_PAIRS and len(pair_targets) != expected_ordered_pairs:
        issues.append(
            f"pair_target_coverage_mismatch: expected {expected_ordered_pairs}, got {len(pair_targets)}"
        )

    diag = rec.get("diagnostics", {})
    coverage = diag.get("object_coverage_ratio")

    if coverage is not None and coverage < 0.5:
        issues.append(f"low_object_coverage_ratio: {coverage:.3f}")

    return issues


def check_output_dir(output_dir: str, mode: str):
    all_issues = []

    coverage_values = []
    object_counts = []
    cluster_object_counts = []
    pair_target_counts = []

    for scene, filename, path in iter_scene_output_files(output_dir, mode):
        data = load_json(path)

        file_issues = []

        for rec in data["paragraphs"]:
            issues = check_paragraph_content(rec)

            diag = rec.get("diagnostics", {})
            coverage = diag.get("object_coverage_ratio")

            if coverage is not None:
                coverage_values.append(coverage)

            object_counts.append(diag.get("num_object_mentions"))
            cluster_object_counts.append(diag.get("source_cluster_num_objects"))
            pair_target_counts.append(diag.get("num_pair_targets"))

            if issues:
                file_issues.append({
                    "paragraph_id": rec["paragraph_id"],
                    "issues": issues,
                    "text": rec["text"],
                    "diagnostics": rec.get("diagnostics", {}),
                })

        print(
            filename,
            "| mode:",
            mode,
            "| paragraphs:",
            data["num_paragraphs"],
            "| paragraphs_with_issues:",
            len(file_issues)
        )

        all_issues.extend(file_issues)

    print(f"\nTotal issues for mode={mode}:", len(all_issues))

    if coverage_values:
        print("\nCoverage summary:")
        print("num paragraphs:", len(coverage_values))
        print("min coverage:", round(min(coverage_values), 3))
        print("max coverage:", round(max(coverage_values), 3))
        print("avg coverage:", round(sum(coverage_values) / len(coverage_values), 3))

    clean_object_counts = [x for x in object_counts if x is not None]
    clean_pair_counts = [x for x in pair_target_counts if x is not None]

    if clean_object_counts:
        print("\nObject mention summary:")
        print("min num_object_mentions:", min(clean_object_counts))
        print("max num_object_mentions:", max(clean_object_counts))
        print("avg num_object_mentions:", round(sum(clean_object_counts) / len(clean_object_counts), 2))

    if clean_pair_counts:
        print("\nPair target summary:")
        print("min num_pair_targets:", min(clean_pair_counts))
        print("max num_pair_targets:", max(clean_pair_counts))
        print("avg num_pair_targets:", round(sum(clean_pair_counts) / len(clean_pair_counts), 2))

    if all_issues:
        from pprint import pprint
        print("\nExample issues:")
        pprint(all_issues[:3])


check_output_dir(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

FloorPlan1_step3_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan2_step3_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan3_step3_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan4_step3_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan5_step3_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan6_step3_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan7_step3_text_diverse.json | mode: diverse | paragraphs: 28 | paragraphs_with_issues: 0
FloorPlan8_step3_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan9_step3_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan10_step3_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan11_step3_text_diverse.json | m

In [ ]:
from pprint import pprint


def load_first_output(output_dir: str, mode: str):
    if not SCENES:
        raise ValueError("SCENES is empty.")

    scene = SCENES[0]
    filename = f"{scene}_step3_text_{mode}.json"
    path = os.path.join(output_dir, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing output file: {path}")

    data = load_json(path)

    return filename, data


sample_file, sample_data = load_first_output(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

print("Sample file:", sample_file)
print("Generation mode:", RUN_MODE)
print("Scene:", sample_data["scene"])
print("Num paragraphs:", sample_data["num_paragraphs"])
print("Num pair targets:", sample_data["num_pair_targets"])

n = min(len(sample_data["paragraphs"]), 5)

for i in range(n):
    rec = sample_data["paragraphs"][i]
    diag = rec["diagnostics"]

    print("\n" + "=" * 100)
    print("Paragraph index:", i)
    print("paragraph_id:", rec["paragraph_id"])
    print("generation_mode:", rec["generation_mode"])

    print("\n[TEXT]")
    print(rec["text"])

    print("\n[coverage]")
    print("source_cluster_num_objects:", diag.get("source_cluster_num_objects"))
    print("num_object_mentions:", diag.get("num_object_mentions"))
    print("object_coverage_ratio:", diag.get("object_coverage_ratio"))
    print("target_object_coverage_ratio:", diag.get("target_object_coverage_ratio"))
    print("max_objects_per_paragraph:", diag.get("max_objects_per_paragraph"))

    print("\n[Explicit relations]")
    for rel in rec["explicit_relations_in_text"]:
        print(
            "-",
            rel["relation"],
            "| template_index:",
            rel["template_index"],
            "| sentence:",
            rel["sentence"]
        )

    print("\n[pair label counts]")
    pprint(diag["pair_label_counts"])

    print("\n[pair evidence counts]")
    pprint(diag["pair_evidence_counts"])

Sample file: FloorPlan1_step3_text_diverse.json
Generation mode: diverse
Scene: FloorPlan1
Num paragraphs: 24
Num pair targets: 2666

Paragraph index: 0
paragraph_id: FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_step3_diverse_0
generation_mode: diverse

[TEXT]
In the described kitchen area, coffee_machine_1 is on counter_top_1. On top of counter_top_1, there is mug_1. You can see pot_1 on counter_top_1. cabinet_1 is above pot_1. cabinet_2 has dish_sponge_1 inside it. cabinet_1 holds wine_bottle_1. counter_top_1 is holding up house_plant_1. On counter_top_1, there is lettuce_1. On counter_top_1, there is paper_towel_roll_1. counter_top_1 is positioned above dish_sponge_1. counter_top_1 sits below wine_bottle_1. On counter_top_1, there is potato_1. dish_sponge_1 is on the left side of pot_1. dish_sponge_1 can be seen to the left of potato_1. house_plant_1 sits to the left of pot_1. paper_towel_roll_1 sits to the left of lettuce_1.

[coverage]
source_cluster_num_objects: 15
num_object_mentio

In [ ]:
from google.colab import files

zip_name = f"/content/pilot_full_step3_{RUN_MODE}_text"

zip_path = shutil.make_archive(
    zip_name,
    "zip",
    OUTPUT_DIR,
)

print("Created:", zip_path)

files.download(zip_path)

Created: /content/pilot_full_step3_diverse_text.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>